In [1]:
# This is a simple code for LBJM-P and LBJM-H algorithms

import numpy as np



XYZ_position_dict = {
    'X':0,
    'Y':1,
    'Z':2,
    0:'X',
    1:'Y',
    2:'Z'
}

# the test two-qubit Pauli strings are [[coefficient_1, coefficient_2,......,coefficient_R],[Pauli_string_1, Pauli_string_2,......,Pauli_string_R]]
# the Hamiltonian is the sum of coefficient_r multiplying by Pauli_string_r, where r in {1, ......, R}.
test_Pauli_strings = [[1,0.5,2],[['X0','X1'],['X0'],['Y0','Z1']]]

# this is local sharpness upper bound (eta^{ub}_{1},eta^{ub}_{2})
test_sharpness_ub = [1,0.8]

In [2]:
# set local sharpness upper bound
def get_eta_ub_collection(sharpness_ub, repetitions):
    eta_ub_collection = [sharpness_ub] * repetitions
    return eta_ub_collection

test_eta_ub = [[1,0.8],[0.9,0.8]]
test_sharpness_ub = test_eta_ub.pop(0)
print(test_eta_ub)
print(test_sharpness_ub)

test_eta_ub0 = get_eta_ub_collection([1,0.8], 3)
print(test_eta_ub0)


[[0.9, 0.8]]
[1, 0.8]
[[1, 0.8], [1, 0.8], [1, 0.8]]


In [3]:
# get one locall unbiased joint measurement based on one local sharpness upper bound
def get_one_locally_unbiased_joint_measurement_eta(n_qubits, eta_ub):
    locally_unbiased_joint_measurement_eta = []
    for i in range(n_qubits):
        local_measurement_eta = np.array([eta_ub[i]/np.sqrt(3)]*3)
        locally_unbiased_joint_measurement_eta.append(local_measurement_eta)
    return np.array([locally_unbiased_joint_measurement_eta])

test_one_locally_unbiased_joint_measurement_eta = get_one_locally_unbiased_joint_measurement_eta(2, test_sharpness_ub)
print(test_one_locally_unbiased_joint_measurement_eta, 0.8/np.sqrt(3))


[[[0.57735027 0.57735027 0.57735027]
  [0.46188022 0.46188022 0.46188022]]] 0.46188021535170065


In [4]:

# from the measurement scheme get EMN for Pauli strings
def get_EMN_collection_from_measurement_eta_Pauli_strings(measurement_etas, Pauli_strings):
    Num_Pauli_strings = len(Pauli_strings[1])
    EMN_collection = [0]*Num_Pauli_strings
    for one_measurement_eta in measurement_etas:
        for one_string in range(Num_Pauli_strings):
            temp_EMN = 1
            for oper_position in Pauli_strings[1][one_string]:
                i_qubit = int(oper_position[1])

                i_oper_position = XYZ_position_dict[oper_position[0]]

                temp_EMN *= (one_measurement_eta[i_qubit][i_oper_position])**2
            EMN_collection[one_string] += temp_EMN
    return EMN_collection

print(get_EMN_collection_from_measurement_eta_Pauli_strings(
    test_one_locally_unbiased_joint_measurement_eta, test_Pauli_strings))


[0.07111111111111114, 0.3333333333333334, 0.07111111111111114]


In [5]:
# check EMN of Pauli strings achieving target EMN or not
def get_score_from_EMN_success_measurement_etas_Pauli_strings(
    EMN_success, measurement_etas, Pauli_strings):
    temp_EMN = get_EMN_collection_from_measurement_eta_Pauli_strings(
        measurement_etas, Pauli_strings)
    score = 0
    for i_th_success, success_YN in enumerate(EMN_success):
        score += (1 - success_YN) * temp_EMN[i_th_success]
    return score

In [6]:
# check EMN again
def check_score_from_EMN_success_measurement_etas_Pauli_strings(
    EMN_success, measurement_etas, Pauli_strings):
    temp_EMN = get_EMN_collection_from_measurement_eta_Pauli_strings(
        measurement_etas, Pauli_strings)
    abs_coeff_collection = np.abs(np.array(Pauli_strings[0]))
    score_coeff_collection = abs_coeff_collection/np.sum(abs_coeff_collection)
    score = 0
    for i_th_success, success_YN in enumerate(EMN_success):
        score += (score_coeff_collection[i_th_success]) ** success_YN * temp_EMN[i_th_success]
    return score

In [7]:
# from known Pauli strings and target EMN, get measurement scheme
def get_measurement_scheme_data_from_Pauli_strings_target_EMN(
    Pauli_strings,n_qubits,target_EMN, eta_ub,old_measurement_scheme = None):
    measurement_scheme_data = {}
    measurement_scheme_data['target_EMN'] = target_EMN
    Num_Pauli_strings = len(target_EMN)
    added_measurement_scheme = []
    added_measurement_eta_scheme = []
    if old_measurement_scheme == None:
        EMN_collection = np.array([0.0] * Num_Pauli_strings)
    else:
        EMN_collection = get_EMN_collection_from_measurement_eta_Pauli_strings(old_measurement_scheme['total_measurement_eta_scheme'], Pauli_strings)

    EMN_success = [0] * Num_Pauli_strings

    
    for i_th_string in range(Num_Pauli_strings):
        if EMN_collection[i_th_string] >= target_EMN[i_th_string]:
            EMN_success[i_th_string] = 1
    EMN_success_sum = np.sum(EMN_success)
    
    while EMN_success_sum < Num_Pauli_strings:
        one_new_measurement_eta = \
            get_one_locally_unbiased_joint_measurement_eta(
                n_qubits = n_qubits, eta_ub=eta_ub)
        one_new_measurement = []
        for i_qubit in range(n_qubits):
            XYZ_eta_collection = np.array([[1,0,0],[0,1,0],[0,0,1]]) * eta_ub[i_qubit]
            XYZ_EMN = []
            for XYZ_eta in XYZ_eta_collection:
                one_new_measurement_eta[0][i_qubit] = XYZ_eta
                temp_score = get_score_from_EMN_success_measurement_etas_Pauli_strings(
                    EMN_success, one_new_measurement_eta, Pauli_strings)
                XYZ_EMN.append(temp_score)
            
            XYZ_choice = XYZ_EMN.index(np.max(XYZ_EMN))
            one_new_measurement_eta[0][i_qubit] = XYZ_eta_collection[XYZ_choice]
        
        for i_qubit in range(n_qubits):
            XYZ_eta_collection = np.array([[1,0,0],[0,1,0],[0,0,1]]) * eta_ub[i_qubit]
            XYZ_EMN = []
            for XYZ_eta in XYZ_eta_collection:
                one_new_measurement_eta[0][i_qubit] = XYZ_eta
                temp_score = check_score_from_EMN_success_measurement_etas_Pauli_strings(
                    EMN_success, one_new_measurement_eta, Pauli_strings)
                XYZ_EMN.append(temp_score)
            XYZ_choice = XYZ_EMN.index(np.max(XYZ_EMN))
            one_new_measurement_eta[0][i_qubit] = XYZ_eta_collection[XYZ_choice]
            
            one_new_measurement.append(XYZ_position_dict[XYZ_choice]+str(i_qubit))
        added_measurement_scheme.append(one_new_measurement)
        added_measurement_eta_scheme.append(one_new_measurement_eta[0])
        
        EMN_added = get_EMN_collection_from_measurement_eta_Pauli_strings(
            one_new_measurement_eta, Pauli_strings)
        EMN_collection += np.array(EMN_added)
        
        for i_th_string in range(Num_Pauli_strings):
            if EMN_collection[i_th_string] >= target_EMN[i_th_string]:
                EMN_success[i_th_string] = 1
        EMN_success_sum = np.sum(EMN_success)
    measurement_scheme_data['EMN_collection'] = EMN_collection
    measurement_scheme_data['added_measurement_scheme'] = added_measurement_scheme
    measurement_scheme_data['added_measurement_eta_scheme'] = added_measurement_eta_scheme
    measurement_scheme_data['added_measurement_num'] = len(added_measurement_scheme)
    if old_measurement_scheme ==None:
        measurement_scheme_data['total_measurement_scheme'] = added_measurement_scheme
        measurement_scheme_data['total_measurement_eta_scheme'] = added_measurement_eta_scheme
        measurement_scheme_data['total_measurement_num'] = len(added_measurement_scheme)
    else:
        measurement_scheme_data['total_measurement_scheme'] = old_measurement_scheme['total_measurement_scheme'] + added_measurement_scheme
        measurement_scheme_data['total_measurement_eta_scheme'] = old_measurement_scheme['total_measurement_eta_scheme'] + added_measurement_eta_scheme
        measurement_scheme_data['total_measurement_num'] = len(measurement_scheme_data['total_measurement_scheme'])
        
    return measurement_scheme_data
                
            
            

In [8]:
# test functions
test_eta_ub2 = get_eta_ub_collection([1,0.8],100)+get_eta_ub_collection([0.9,0.8],100)

test_measurement_scheme = get_measurement_scheme_data_from_Pauli_strings_target_EMN(
    test_Pauli_strings,2,[50, 50, 50], test_sharpness_ub)

test_measurement_scheme1 = get_measurement_scheme_data_from_Pauli_strings_target_EMN(
    test_Pauli_strings,2,[50, 100, 50], test_sharpness_ub,test_measurement_scheme)


In [9]:
print(test_measurement_scheme['total_measurement_scheme'])
print(test_measurement_scheme['total_measurement_eta_scheme'][0])
print(test_measurement_scheme['EMN_collection'])
print(test_measurement_scheme['total_measurement_num'])

print(test_measurement_scheme1['added_measurement_scheme'])
print(test_measurement_scheme1['total_measurement_num'])
print(test_measurement_scheme1['EMN_collection'])

[['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0', 'X1'], ['X0'

In [10]:
# from Pauli strings, measurement budger and possible already existed EMN, get new target EMN

def get_target_EMN_collection(Pauli_strings, N_budget, initial_EMN_collection = 0):
    
    coeff_collection = Pauli_strings[0]
    
    
    relative_coeff = np.abs(np.array(coeff_collection))/np.sum(np.abs(np.array(coeff_collection)))
    
    idea_EMN_sum = np.sum(initial_EMN_collection) + N_budget
    
    target_EMN_collection = idea_EMN_sum * relative_coeff
    
    added_EMN_collection = target_EMN_collection - initial_EMN_collection
    for i_EMN, one_EMN in enumerate(added_EMN_collection):
        if one_EMN <0:
            added_EMN_collection[i_EMN] = 0
    return added_EMN_collection



In [11]:
# from the given measurement budget and known Pauli strings, get measurement scheme
def get_measurement_scheme_data_from_Pauli_strings_N_budget(
    Pauli_strings, n_qubits, N_budget, eta_ub, old_measurement_scheme_data=None):
    measurement_scheme_data = {}
    coeff_collection = np.abs(np.array(Pauli_strings[0]))
    Num_Pauli_strings = len(coeff_collection)
    proportion_coeff_collection = coeff_collection/np.sum(coeff_collection)
    measurement_scheme_data['target_propor'] = proportion_coeff_collection/np.min(proportion_coeff_collection)
    temp_N_budget = N_budget
    added_measurement_scheme = []
    added_measurement_eta_scheme = []
    if old_measurement_scheme_data==None:
        old_EMN_collection = np.array([0.0]*Num_Pauli_strings)
    else:
        old_EMN_collection = get_EMN_collection_from_measurement_eta_Pauli_strings(old_measurement_scheme_data['total_measurement_eta_scheme'], Pauli_strings)
    added_EMN_collection = get_target_EMN_collection(Pauli_strings, N_budget, initial_EMN_collection = old_EMN_collection)
    while  temp_N_budget >= 0:
        
        temp_measurement_scheme_data = get_measurement_scheme_data_from_Pauli_strings_target_EMN(
            Pauli_strings, n_qubits, added_EMN_collection, eta_ub, old_measurement_scheme_data)
        added_measurement_scheme += temp_measurement_scheme_data['added_measurement_scheme']
        added_measurement_eta_scheme += temp_measurement_scheme_data['added_measurement_eta_scheme']
        old_EMN_collection += temp_measurement_scheme_data['EMN_collection']
        temp_N_budget -= temp_measurement_scheme_data['added_measurement_num']
        added_EMN_collection = get_target_EMN_collection(
            Pauli_strings, temp_N_budget, old_EMN_collection)
        
    
    final_added_measurement_scheme = added_measurement_scheme[:N_budget]
    final_added_measurement_eta_scheme = added_measurement_eta_scheme[:N_budget]
    
    
    measurement_scheme_data['added_measurement_scheme'] = final_added_measurement_scheme
    measurement_scheme_data['added_measurement_num'] = len(final_added_measurement_scheme)
    measurement_scheme_data['added_measurement_eta_scheme'] = final_added_measurement_eta_scheme
    final_added_EMN_collection = get_EMN_collection_from_measurement_eta_Pauli_strings(
        final_added_measurement_eta_scheme, Pauli_strings)
    measurement_scheme_data['added_EMN_collection'] = final_added_EMN_collection
    if old_measurement_scheme_data==None:
        measurement_scheme_data['total_measurement_scheme'] = measurement_scheme_data['added_measurement_scheme']
        measurement_scheme_data['total_measurement_num'] = measurement_scheme_data['added_measurement_num']
        measurement_scheme_data['total_measurement_eta_scheme'] = measurement_scheme_data['added_measurement_eta_scheme']
        measurement_scheme_data['total_EMN_collection'] = measurement_scheme_data['added_EMN_collection']
    else:
        measurement_scheme_data['total_measurement_scheme'] = measurement_scheme_data['added_measurement_scheme'] + old_measurement_scheme_data['total_measurement_scheme']
        measurement_scheme_data['total_measurement_num'] = len(measurement_scheme_data['total_measurement_scheme'])
        measurement_scheme_data['total_measurement_eta_scheme'] = measurement_scheme_data['added_measurement_eta_scheme']+ old_measurement_scheme_data['total_measurement_eta_scheme']
        measurement_scheme_data['total_EMN_collection'] = get_EMN_collection_from_measurement_eta_Pauli_strings(
            old_measurement_scheme_data['total_measurement_eta_scheme'], Pauli_strings)
    
    check_propor = [0]*Num_Pauli_strings
    for i_th_Pauli_string in range(Num_Pauli_strings):
        check_propor[i_th_Pauli_string] = measurement_scheme_data['total_EMN_collection'][i_th_Pauli_string]/proportion_coeff_collection[i_th_Pauli_string]
    final_check_propor = np.array(check_propor)/np.min(check_propor)
    measurement_scheme_data['check_propor'] = final_check_propor
    return measurement_scheme_data



In [12]:
# test LBJM-P algorithm
test_Pauli_strings1 = [[1,0.5,2],[['X0','X1'],['X0'],['Y0','Z1']]]
test_eta_ub3 = get_eta_ub_collection([1,0.8],200)+get_eta_ub_collection([0.9,0.8],100)
print(len(test_eta_ub3))
test_measurement_scheme2 = get_measurement_scheme_data_from_Pauli_strings_N_budget(
    test_Pauli_strings1, 2, 300, test_sharpness_ub,test_measurement_scheme1)

300


In [13]:
print(test_measurement_scheme2['total_EMN_collection'])
print(test_measurement_scheme2['total_measurement_num'])
print(test_measurement_scheme2['target_propor'])
print(test_measurement_scheme2['check_propor'])

[64.00000000000004, 100.0, 50.56000000000004]
479
[2. 1. 4.]
[2.53164557 7.91139241 1.        ]


In [14]:
# sort Pauli strings from maximal to minimal absolute values of coefficients
def sort_Pauli_strings(Pauli_strings):
    coeff_collection = Pauli_strings[0]
    Pauli_string_oper = Pauli_strings[1]
    Num_Pauli_strings = len(coeff_collection)
    sort_coeff_collection = []
    sort_Pauli_operators = []
    while Num_Pauli_strings!=0:
        abs_coeff_collection = np.abs(np.array(coeff_collection))
        max_index = np.argmax(abs_coeff_collection)
        sort_coeff_collection.append(coeff_collection[max_index])
        sort_Pauli_operators.append(Pauli_string_oper[max_index])
        coeff_collection.remove(coeff_collection[max_index])
        Pauli_string_oper.remove(Pauli_string_oper[max_index])
        Num_Pauli_strings = len(coeff_collection)
    return [sort_coeff_collection, sort_Pauli_operators]
        
test_sort_Pauli_strings = sort_Pauli_strings(test_Pauli_strings1)
print(test_sort_Pauli_strings)

[[2, 1, 0.5], [['Y0', 'Z1'], ['X0', 'X1'], ['X0']]]


In [15]:
# LBJM-H algorithm will devide the Hamiltonian into serval sub-Hamiltonians, where one sub-Hamiltion can be estimated by one measurement
def find_one_estimator_for_Pauli_strings(Pauli_strings, n_qubits):
    rest_Pauli_strings = Pauli_strings
    noiseless_eta = [1.0]*n_qubits
    one_compatible_mea_eta = get_one_locally_unbiased_joint_measurement_eta(n_qubits, noiseless_eta)
    XYZ_eta = np.array([[1,0,0],[0,1,0],[0,0,1]])
    for i_qubit in range(n_qubits):
        temp_score_collection = []
        for j_position in XYZ_eta:
            one_compatible_mea_eta[0][i_qubit] = j_position
            temp_ENM = get_EMN_collection_from_measurement_eta_Pauli_strings(one_compatible_mea_eta, rest_Pauli_strings)
            temp_score_collection.append(np.sum(temp_ENM))
        max_Pauli = temp_score_collection.index(max(temp_score_collection))
        one_compatible_mea_eta[0][i_qubit] = XYZ_eta[max_Pauli]
    
    EMN_collection = get_EMN_collection_from_measurement_eta_Pauli_strings(one_compatible_mea_eta, rest_Pauli_strings)
    New_rest_Pauli_strings = rest_Pauli_strings[1].copy()
    New_rest_coeff_collection = rest_Pauli_strings[0].copy()
    estimator_coeff_collection = []
    estimator_Pauli_collection = []
    for m_string, EMN in enumerate(EMN_collection):
        if EMN != 0:
            estimator_Pauli_collection.append(rest_Pauli_strings[1][m_string])
            New_rest_Pauli_strings.remove(rest_Pauli_strings[1][m_string])
            estimator_coeff_collection.append(rest_Pauli_strings[0][m_string])
            New_rest_coeff_collection.remove(rest_Pauli_strings[0][m_string])
    one_estimator_data = {}
    temp_Pauli_XYZ = []
    for i_qubit, i_position in enumerate(one_compatible_mea_eta[0]):
        if all(i_position == np.array([1,0,0])):
            temp_Pauli_XYZ.append('X'+str(i_qubit))
        if all(i_position == np.array([0,1,0])):
            temp_Pauli_XYZ.append('Y'+str(i_qubit))
        if all(i_position == np.array([0,0,1])):
            temp_Pauli_XYZ.append('Z'+str(i_qubit))
    
    one_estimator_data['one_compatible_mea'] = temp_Pauli_XYZ
    one_estimator_data['estimator_Pauli_strings'] = [estimator_coeff_collection, estimator_Pauli_collection]
    one_estimator_data['New_rest_Pauli_strings'] = [New_rest_coeff_collection, New_rest_Pauli_strings]
    return one_estimator_data

# find all sub-Hamiltonians
def find_estimator_measurements_for_Pauli_strings(Pauli_strings, n_qubits):
    rest_Pauli_strings = Pauli_strings.copy()
    estimator_measurements_data = []
    while len(rest_Pauli_strings[1]) !=0 :
        one_estimator_data = find_one_estimator_for_Pauli_strings(rest_Pauli_strings, n_qubits=n_qubits)
        estimator_measurements_data.append(one_estimator_data)
        rest_Pauli_strings = one_estimator_data['New_rest_Pauli_strings']
    return estimator_measurements_data

test_estimator_measurements = find_estimator_measurements_for_Pauli_strings(test_Pauli_strings, 2)
print(test_estimator_measurements)

# after finding sub-Hamiltonians, the bound of every sub-Hamiltonian should be indetified
def get_bound_for_estimator_measurements_data(estimator_measurements_data, eta_ub):
    for one_estimator in estimator_measurements_data:
        one_estimator_mea = one_estimator['one_compatible_mea']
        one_estimator_PS = one_estimator['estimator_Pauli_strings']
        n_qubits = len(one_estimator_mea)
        one_estimator_outcomes = []
        for one_outcome in range(2**n_qubits):
            bi_outcome = bin(one_outcome)[2:].zfill(n_qubits)
            temp_outcome = 0
            for m_string in range(len(one_estimator_PS[1])):
                m_coeff = one_estimator_PS[0][m_string]
                m_Pauli_observable = one_estimator_PS[1][m_string]
                temp_v_eta = 1
                
                for oper_position in m_Pauli_observable:
                    position = int(oper_position[1])
                    m_eta = eta_ub[position]
                    temp_v_eta *= (-1)**int(bi_outcome[position])/m_eta
                temp_outcome += m_coeff * temp_v_eta
            one_estimator_outcomes.append(temp_outcome)
        one_estimator_max_v = np.max(one_estimator_outcomes)
        one_estimator_min_v = np.min(one_estimator_outcomes)
        one_estimator_bound = np.abs(one_estimator_max_v-one_estimator_min_v)/2
        one_estimator['bound'] = one_estimator_bound
    return estimator_measurements_data

test_eta_ub4 = [1,1]
test_bound = get_bound_for_estimator_measurements_data(test_estimator_measurements, test_eta_ub4)
print(test_bound)

# At the beginning, the measurement scheme is filled with unbiased LBJMs, so it is necessary to find the bounds of unbiased LBJMs estimating sub-Hamiltonians
def get_bound_from_LUBJM(Pauli_strings, eta_ub):
    n_qubits = len(eta_ub)
    LUBJM_estimator_outcomes = []
    for one_outcome in range(2**n_qubits):
        bi_outcome = bin(one_outcome)[2:].zfill(n_qubits)
        temp_outcome = 0
        for m_string in range(len(Pauli_strings[0])):
            m_coeff = Pauli_strings[0][m_string]
            m_Pauli_observable = Pauli_strings[1][m_string]
            temp_v_eta = 1
            
            for oper_position in m_Pauli_observable:
                position = int(oper_position[1])
                m_eta = eta_ub[position]/np.sqrt(3)
                temp_v_eta *= (-1)**int(bi_outcome[position])/m_eta
            temp_outcome += m_coeff * temp_v_eta
        LUBJM_estimator_outcomes.append(temp_outcome)
    LUBJM_estimator_max_v = np.max(LUBJM_estimator_outcomes)
    LUBJM_estimator_min_v = np.min(LUBJM_estimator_outcomes)
    LUBJM_estimator_bound = (LUBJM_estimator_max_v-LUBJM_estimator_min_v)/2
    return LUBJM_estimator_bound
# EMN for the total Hamiltonian
def get_score_from_bounds_measurement_num(bounds, measurement_num):
    length = len(bounds)
    score = 0
    for i_bound in range(length):
        score += bounds[i_bound] ** 2/measurement_num[i_bound]
    final_score = 1/score
    return final_score
# from known Pauli strings, local sharpness upper bounds, sub-Hamiltonians and measurement budget, get LBJM-H measurement scheme.
# The results are joint measurement of sub-Hamiltonians and measurement repetitions allocated for them.
def get_measurement_scheme_data_from_bound_data(Pauli_strings, estimator_measurements_data, total_N, eta_ub):
    LUBJM_estimator_bound = get_bound_from_LUBJM(Pauli_strings, eta_ub)
    bound_collection = []
    for one_estimator_data in estimator_measurements_data:
        bound_collection.append(one_estimator_data['bound'])
    bound_collection + [LUBJM_estimator_bound]
    bound_collection = np.array(bound_collection)
    measurement_num = [1]*len(bound_collection) + [total_N-len(bound_collection)]
    for one_mea_num in range(total_N-len(bound_collection)):
        temp_score_collection = []
        for one_mea in range(len(measurement_num)):
            measurement_num[one_mea] += 1
            temp_score = get_score_from_bounds_measurement_num(bound_collection, measurement_num)
            temp_score_collection.append(temp_score)
            measurement_num[one_mea] -= 1
        max_score_index = temp_score_collection.index(max(temp_score_collection))
        measurement_num[max_score_index] += 1
        measurement_num[len(bound_collection)]-=1
    return measurement_num

test_mea_num = get_measurement_scheme_data_from_bound_data(test_Pauli_strings,test_bound, 100, test_sharpness_ub)
print(test_mea_num)

[{'one_compatible_mea': ['X0', 'X1'], 'estimator_Pauli_strings': [[1, 0.5], [['X0', 'X1'], ['X0']]], 'New_rest_Pauli_strings': [[2], [['Y0', 'Z1']]]}, {'one_compatible_mea': ['Y0', 'Z1'], 'estimator_Pauli_strings': [[2], [['Y0', 'Z1']]], 'New_rest_Pauli_strings': [[], []]}]
[{'one_compatible_mea': ['X0', 'X1'], 'estimator_Pauli_strings': [[1, 0.5], [['X0', 'X1'], ['X0']]], 'New_rest_Pauli_strings': [[2], [['Y0', 'Z1']]], 'bound': 1.5}, {'one_compatible_mea': ['Y0', 'Z1'], 'estimator_Pauli_strings': [[2], [['Y0', 'Z1']]], 'New_rest_Pauli_strings': [[], []], 'bound': 2.0}]
[43, 57, 0]
